# Voice-to-Voice Agent with Eden AI

A push-to-talk voice agent that chains three Eden AI features through one API key:

**Speech-to-Text → LLM → Text-to-Speech**

Click record, speak, click stop, then send. The audio is uploaded, transcribed, sent to an LLM with conversation history, and the answer is spoken back. Swap any provider in one line at the top of the config cell.

**Prerequisites:** an Eden AI API key (set as `EDENAI_API_KEY` env var), a microphone, and a browser-based Jupyter frontend (Classic Notebook or JupyterLab — VS Code's Jupyter extension does not expose mic permissions reliably).

In [ ]:
%pip install --quiet requests ipywebrtc ipywidgets

## 1. Configuration

All provider/model strings live here — swap them to compare providers without touching the rest of the notebook.

In [ ]:
import os

EDENAI_API_KEY = os.environ.get("EDENAI_API_KEY", "YOUR_EDEN_AI_API_KEY")
EDENAI_BASE = "https://api.edenai.run/v3"

STT_MODEL = "audio/speech_to_text_async/openai"
LLM_MODEL = "anthropic/claude-sonnet-4-5"
TTS_MODEL = "audio/tts/openai/tts-1"

LANGUAGE = "en"
SYSTEM_PROMPT = "You are a concise voice assistant. Keep replies under two short sentences."

HEADERS = {"Authorization": f"Bearer {EDENAI_API_KEY}"}
JSON_HEADERS = {**HEADERS, "Content-Type": "application/json"}

## 2. Eden AI helpers

Four small functions wrap the three Eden AI endpoints used by the pipeline:

- `upload_file` → `POST /v3/upload` (multipart) — returns a `file_id` valid for 7 days
- `transcribe` → `POST /v3/universal-ai/async` then poll `GET /v3/universal-ai/async/{job_id}`
- `chat` → `POST /v3/llm/chat/completions` (OpenAI-compatible)
- `synthesize` → `POST /v3/universal-ai` (sync) returns an `audio_resource_url` we then download

In [ ]:
import time
import requests


def upload_file(audio_bytes: bytes, filename: str = "recording.webm") -> str:
    files = {"file": (filename, audio_bytes, "audio/webm")}
    data = {"purpose": "speech"}
    r = requests.post(f"{EDENAI_BASE}/upload", headers=HEADERS, files=files, data=data)
    r.raise_for_status()
    return r.json()["file_id"]


def transcribe(file_id: str, language: str = LANGUAGE) -> str:
    payload = {
        "model": STT_MODEL,
        "input": {"file": file_id, "language": language},
    }
    r = requests.post(f"{EDENAI_BASE}/universal-ai/async", headers=JSON_HEADERS, json=payload)
    r.raise_for_status()
    job_id = r.json()["job_id"]

    while True:
        time.sleep(1)
        s = requests.get(f"{EDENAI_BASE}/universal-ai/async/{job_id}", headers=HEADERS)
        s.raise_for_status()
        data = s.json()
        status = data.get("status")
        if status == "finished":
            return data["result"]["text"]
        if status == "failed":
            raise RuntimeError(data.get("error_message", "STT job failed"))


def chat(messages: list) -> str:
    payload = {"model": LLM_MODEL, "messages": messages}
    r = requests.post(f"{EDENAI_BASE}/llm/chat/completions", headers=JSON_HEADERS, json=payload)
    r.raise_for_status()
    return r.json()["choices"][0]["message"]["content"]


def synthesize(text: str) -> bytes:
    payload = {"model": TTS_MODEL, "input": {"text": text}}
    r = requests.post(f"{EDENAI_BASE}/universal-ai", headers=JSON_HEADERS, json=payload)
    r.raise_for_status()
    audio_url = r.json()["audio_resource_url"]
    return requests.get(audio_url).content

## 3. Push-to-talk widget

`ipywebrtc.AudioRecorder` gives us in-browser mic capture. Click ● to record, ■ to stop, then **Send to agent**.

Conversation history accumulates across rounds, so the agent remembers what you said before.

In [ ]:
from ipywebrtc import AudioRecorder, CameraStream
from ipywidgets import Button, Output, VBox
from IPython.display import Audio, display, clear_output

stream = CameraStream(constraints={"audio": True, "video": False})
recorder = AudioRecorder(stream=stream)

send_btn = Button(description="Send to agent", button_style="primary")
reset_btn = Button(description="Reset conversation")
log = Output()
audio_out = Output()

history = [{"role": "system", "content": SYSTEM_PROMPT}]


def run_pipeline(audio_bytes: bytes) -> None:
    with log:
        print("Uploading audio…")
    file_id = upload_file(audio_bytes)

    with log:
        print("Transcribing…")
    user_text = transcribe(file_id)

    with log:
        print(f"🗣️  You: {user_text}")

    history.append({"role": "user", "content": user_text})

    with log:
        print("Thinking…")
    reply = chat(history)
    history.append({"role": "assistant", "content": reply})

    with log:
        print(f"🤖 Agent: {reply}")
        print("Synthesizing speech…")
    audio = synthesize(reply)

    with audio_out:
        clear_output(wait=True)
        display(Audio(audio, autoplay=True))


def on_send(_):
    audio_bytes = recorder.audio.value
    if not audio_bytes:
        with log:
            print("(record something first)")
        return
    try:
        run_pipeline(audio_bytes)
    except Exception as e:
        with log:
            print(f"Error: {e}")


def on_reset(_):
    history.clear()
    history.append({"role": "system", "content": SYSTEM_PROMPT})
    with log:
        clear_output()
        print("Conversation reset.")


send_btn.on_click(on_send)
reset_btn.on_click(on_reset)

display(VBox([recorder, send_btn, reset_btn, log, audio_out]))

## 4. Try swapping providers

Edit the constants at the top, re-run the config cell, and keep talking. Some combinations to try:

| Goal | Suggested model |
|---|---|
| Cheap STT | `audio/speech_to_text_async/deepgram/nova-3` |
| Higher voice quality | `audio/tts/elevenlabs/eleven_turbo_v2_5` |
| Cheap multilingual TTS | `audio/tts/google/wavenet` |
| Different reasoning | `openai/gpt-4`, `deepseek/deepseek-chat`, `google/gemini-2.5-flash` |

## 5. Variant: voice translator

Replace the LLM step in `run_pipeline` with a translation prompt and pick a multilingual TTS:

```python
TTS_MODEL = "audio/tts/elevenlabs/eleven_multilingual_v2"

def translate(text: str, target: str = "French") -> str:
    return chat([
        {"role": "system", "content": f"Translate the user's text to {target}. Output only the translation."},
        {"role": "user", "content": text},
    ])

# in run_pipeline, replace `reply = chat(history)` with:
# reply = translate(user_text, target="French")
```

Same three-feature pipeline, completely different application.